# BetaVAE Beta Sweep Analysis
Train: 2013–2023 | Test: 2024–2026 | K=8 latent factors

Run `python scripts/run_beta_sweep.py` before opening this notebook.

In [ ]:
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import seaborn as sns

SWEEP   = next(p / "beta_sweep_k8" for p in [Path(".").resolve(), Path(".").resolve().parent] if (p / "beta_sweep_k8").exists())  # run from results/ or repo root
FACTORS = SWEEP / "factors"
ERRORS  = SWEEP / "errors"

assert ERRORS.exists(), (
    "No beta sweep data found. Run first:\n"
    "  python scripts/run_beta_sweep.py"
)

def _parse_beta(name):
    m = re.search(r"BetaVAE_b([\d.]+)", name)
    return float(m.group(1)) if m else None

# Auto-discover beta variants from saved CSVs
BETA_ARCHS = sorted(
    {f.stem.replace("_train_recon_error", "").replace("_test_recon_error", "")
     for f in ERRORS.glob("BetaVAE_b*_recon_error.csv")},
    key=_parse_beta,
)
BETAS = [_parse_beta(a) for a in BETA_ARCHS]
assert BETA_ARCHS, "No BetaVAE_b* CSV files found in beta_sweep/errors/"
print(f"Found {len(BETA_ARCHS)} beta variants: {BETAS}")

# Load recon errors
train_errors = {
    name: pd.read_csv(ERRORS / f"{name}_train_recon_error.csv", index_col=0, parse_dates=True)
    for name in BETA_ARCHS
}
test_errors = {
    name: pd.read_csv(ERRORS / f"{name}_test_recon_error.csv", index_col=0, parse_dates=True)
    for name in BETA_ARCHS
}

# Per-commodity MSE tables
def mse_table(errors_dict):
    rows = {}
    for name, err in errors_dict.items():
        per_com = (err ** 2).mean()
        per_com["ALL"] = float((err.to_numpy() ** 2).mean())
        rows[name] = per_com
    return pd.DataFrame(rows).T

train_mse_tbl = mse_table(train_errors)
test_mse_tbl  = mse_table(test_errors)

train_mses = [train_mse_tbl.loc[a, "ALL"] for a in BETA_ARCHS]
test_mses  = [test_mse_tbl.loc[a, "ALL"]  for a in BETA_ARCHS]

## Summary Table

In [ ]:
summary = pd.DataFrame({
    "beta":           BETAS,
    "train_recon_mse": train_mses,
    "test_recon_mse":  test_mses,
    "generalisation_gap": [t - r for r, t in zip(train_mses, test_mses)],
}, index=BETA_ARCHS)
display(summary.round(5))

## 1. Reconstruction MSE vs Beta

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(BETAS, train_mses, "o-", label="Train", color="steelblue")
ax.plot(BETAS, test_mses,  "s-", label="Test",  color="coral")
ax.set_xlabel("Beta")
ax.set_ylabel("Reconstruction MSE (standardised units)")
ax.set_title("BetaVAE: reconstruction MSE vs beta")
ax.set_xscale("log")
ax.set_xticks(BETAS)
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.legend()
fig.tight_layout()
plt.show()

## 2. Generalisation Gap vs Beta

In [ ]:
gaps = [t - r for r, t in zip(train_mses, test_mses)]
colors = ["steelblue" if g >= 0 else "crimson" for g in gaps]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([str(b) for b in BETAS], gaps, color=colors)
ax.axhline(0, color="k", lw=0.8, ls="--")
ax.set_xlabel("Beta")
ax.set_ylabel("Test MSE − Train MSE")
ax.set_title("Generalisation gap (test − train MSE) vs beta")
fig.tight_layout()
plt.show()

## 3. Per-Commodity MSE Heatmap

In [ ]:
beta_labels = [f"β={b}" for b in BETAS]

for split, tbl in (("train", train_mse_tbl), ("test", test_mse_tbl)):
    data = tbl.drop(columns="ALL").rename(index=dict(zip(BETA_ARCHS, beta_labels)))
    fig, ax = plt.subplots(figsize=(16, 5))
    sns.heatmap(
        data, cmap="YlOrRd", annot=True, fmt=".2f",
        annot_kws={"size": 7}, cbar_kws={"label": "Reconstruction MSE"}, ax=ax,
    )
    ax.set_title(f"Per-commodity {split} reconstruction MSE vs beta")
    ax.set_xlabel("Commodity")
    ax.set_ylabel("Beta")
    fig.tight_layout()
    plt.show()

## 4. Factor Variances vs Beta

Low variance on a factor means the encoder has collapsed that dimension toward zero — the KL penalty at high beta forces the latent code toward the prior, killing information.

In [ ]:
def load_factor_var(arch, split):
    return pd.read_csv(FACTORS / f"{arch}_{split}_factors.csv", index_col=0, parse_dates=True).var()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, split in zip(axes, ("train", "test")):
    var_matrix = pd.DataFrame(
        {f"β={b}": load_factor_var(a, split) for a, b in zip(BETA_ARCHS, BETAS)}
    ).T
    sns.heatmap(
        var_matrix, cmap="YlOrRd", annot=True, fmt=".2f",
        annot_kws={"size": 9}, ax=ax, cbar_kws={"label": "Variance"},
    )
    ax.set_title(f"Factor variances — {split} set")
    ax.set_xlabel("Factor")
    ax.set_ylabel("Beta")
fig.tight_layout()
plt.show()

## 5. Active Factor Count vs Beta

A factor is "active" if its variance exceeds a threshold (0.1). Shows how many dimensions the encoder uses at each beta.

In [ ]:
THRESHOLD = 0.1

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, split in zip(axes, ("train", "test")):
    active_counts = [
        int((load_factor_var(a, split) > THRESHOLD).sum())
        for a in BETA_ARCHS
    ]
    ax.bar([str(b) for b in BETAS], active_counts, color="steelblue")
    ax.set_ylim(0, max(active_counts) + 1)
    ax.set_xlabel("Beta")
    ax.set_ylabel(f"Active factors (variance > {THRESHOLD})")
    ax.set_title(f"Active factor count — {split} set")
fig.tight_layout()
plt.show()

## 6. Factor Time Series

In [ ]:
for split in ("train", "test"):
    n = len(BETA_ARCHS)
    fig, axes = plt.subplots(n, 1, figsize=(14, 3 * n), sharex=True)
    if n == 1:
        axes = [axes]
    for ax, (arch, beta) in zip(axes, zip(BETA_ARCHS, BETAS)):
        factors = pd.read_csv(FACTORS / f"{arch}_{split}_factors.csv", index_col=0, parse_dates=True)
        for col in factors.columns:
            ax.plot(factors.index, factors[col], lw=0.8, alpha=0.8, label=col)
        ax.set_title(f"β={beta}")
        ax.set_ylabel("Factor value")
        ax.legend(loc="upper right", fontsize=7, ncol=5)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    year_range = "2013–2023" if split == "train" else "2024–2026"
    fig.suptitle(f"Latent factor time series — {split} set ({year_range})", fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

## 7. R² Explained per Beta

R² = 1 − MSE (valid because input is z-scored, so baseline variance = 1 per commodity).  
RMSE = √MSE.  
Higher β → stronger KL penalty → factors compressed toward prior → reconstruction worsens → lower R².

In [ ]:
_SECTOR_MAP = {
    "Brent":"Energy","WTI":"Energy","NaturalGas":"Energy","Gasoline":"Energy",
    "Diesel":"Energy","HeatingOil":"Energy","ThermalCoal":"Energy","Methanol":"Energy",
    "Gold":"Metals","Silver":"Metals","Copper":"Metals","Aluminium":"Metals",
    "Nickel":"Metals","Zinc":"Metals","Platinum":"Metals","HRCSteel":"Metals",
    "SGXIronOre":"Metals","Lithium":"Metals",
    "Coffee":"Agriculture","Corn":"Agriculture","Cotton":"Agriculture",
    "LeanHogs":"Agriculture","LiveCattle":"Agriculture","Sugar":"Agriculture",
    "Soybeans":"Agriculture","SoybeanOil":"Agriculture","HRWWheat":"Agriculture",
}
_SECTOR_ORDER = ["Energy", "Metals", "Agriculture"]
beta_labels   = [f"β={b}" for b in BETAS]

def _metrics_df(mse_tbl):
    rows = []
    for arch in BETA_ARCHS:
        for com in mse_tbl.drop(columns="ALL").columns:
            mse = mse_tbl.loc[arch, com]
            rows.append({
                "beta_arch": arch,
                "commodity": com,
                "sector":    _SECTOR_MAP.get(com, "?"),
                "MSE":       mse,
                "RMSE":      np.sqrt(mse),
                "R2":        1.0 - mse,
            })
    return pd.DataFrame(rows)

train_metrics = _metrics_df(train_mse_tbl)
test_metrics  = _metrics_df(test_mse_tbl)

for label, df in [("Train (2013–2023)", train_metrics), ("Test (2024–2026)", test_metrics)]:
    print(f"\n{'═'*60}")
    print(f"  {label}")
    print(f"{'═'*60}")

    overall = df.groupby("beta_arch")[["MSE","RMSE","R2"]].mean().loc[BETA_ARCHS]
    overall.index = beta_labels
    print("\n— Overall —")
    display(overall.round(4))

    sector_tbl = (
        df.groupby(["beta_arch","sector"])[["MSE","RMSE","R2"]]
        .mean()
        .unstack("sector")
        .loc[BETA_ARCHS]
        .reindex(columns=pd.MultiIndex.from_product([["MSE","RMSE","R2"], _SECTOR_ORDER]))
    )
    sector_tbl.index = beta_labels
    print("\n— Per sector —")
    display(sector_tbl.round(4))

In [ ]:
_COMMODITIES_ORDERED = sorted(
    _SECTOR_MAP,
    key=lambda c: (_SECTOR_ORDER.index(_SECTOR_MAP[c]), list(_SECTOR_MAP).index(c)),
)

for label, metrics in [("Train (2013–2023)", train_metrics), ("Test (2024–2026)", test_metrics)]:
    r2_wide = (
        metrics.pivot(index="beta_arch", columns="commodity", values="R2")
        .loc[BETA_ARCHS, _COMMODITIES_ORDERED]
    )
    r2_wide.index = beta_labels
    fig, ax = plt.subplots(figsize=(22, 4))
    sns.heatmap(
        r2_wide,
        cmap="RdYlGn", center=0, vmin=-0.5, vmax=1.0,
        annot=True, fmt=".2f", annot_kws={"size": 6},
        cbar_kws={"label": "R²  (1 = perfect, 0 = baseline, < 0 = worse than mean)"},
        ax=ax,
    )
    ax.set_title(f"Per-commodity R² — {label}", fontsize=12)
    ax.set_xlabel("Commodity  (Energy → Metals → Agriculture)", fontsize=9)
    ax.set_ylabel("Beta")
    ax.tick_params(axis="x", rotation=90, labelsize=7)
    fig.tight_layout()
    plt.show()

## 8. Factor Correlation Heatmaps

In [ ]:
n = len(BETA_ARCHS)
ncols = 2
nrows = (n + 1) // ncols

for split in ("train", "test"):
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4 * nrows))
    axes = axes.flatten()
    for ax, (arch, beta) in zip(axes, zip(BETA_ARCHS, BETAS)):
        factors = pd.read_csv(FACTORS / f"{arch}_{split}_factors.csv", index_col=0, parse_dates=True)
        sns.heatmap(
            factors.corr(), ax=ax, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            annot=True, fmt=".2f", annot_kws={"size": 8}, square=True, cbar=False,
        )
        ax.set_title(f"β={beta}")
    for ax in axes[n:]:
        ax.set_visible(False)
    fig.suptitle(f"Factor cross-correlations — {split} set", fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

## 9. Latent Factor Interpretation

For each beta value, we compute the **correlation between each latent factor and each commodity return** — the autoencoder equivalent of PCA loadings. Higher β values are expected to produce more disentangled but less commodity-specific factors as the KL penalty pushes the posterior toward a structureless prior.

In [ ]:
returns_raw = pd.read_csv("../data/returns.csv", index_col=0, parse_dates=True)

SECTOR_MAP = {
    "Brent":"Energy","WTI":"Energy","NaturalGas":"Energy","Gasoline":"Energy",
    "Diesel":"Energy","HeatingOil":"Energy","ThermalCoal":"Energy","Methanol":"Energy",
    "Gold":"Metals","Silver":"Metals","Copper":"Metals","Aluminium":"Metals",
    "Nickel":"Metals","Zinc":"Metals","Platinum":"Metals","HRCSteel":"Metals",
    "SGXIronOre":"Metals","Lithium":"Metals",
    "Coffee":"Agriculture","Corn":"Agriculture","Cotton":"Agriculture",
    "LeanHogs":"Agriculture","LiveCattle":"Agriculture","Sugar":"Agriculture",
    "Soybeans":"Agriculture","SoybeanOil":"Agriculture","HRWWheat":"Agriculture",
}
SECTOR_ORDER = ["Energy", "Metals", "Agriculture"]
COMMODITIES_ORDERED = sorted(
    SECTOR_MAP, key=lambda c: (SECTOR_ORDER.index(SECTOR_MAP[c]), list(SECTOR_MAP).index(c))
)

def factor_commodity_corr(arch, split):
    factors = pd.read_csv(FACTORS / f"{arch}_{split}_factors.csv", index_col=0, parse_dates=True)
    rets    = returns_raw.reindex(factors.index)[COMMODITIES_ORDERED]
    return pd.DataFrame(
        {col: factors.corrwith(rets[col]) for col in rets.columns},
        index=factors.columns,
    )

### 9a. Factor–commodity correlation heatmaps (train then test)

In [ ]:
for split in ("train", "test"):
    nb = len(BETA_ARCHS)
    fig, axes = plt.subplots(nb, 1, figsize=(24, 4 * nb))
    if nb == 1:
        axes = [axes]
    for ax, (arch, beta) in zip(axes, zip(BETA_ARCHS, BETAS)):
        corr = factor_commodity_corr(arch, split)
        sns.heatmap(
            corr, ax=ax, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            annot=True, fmt=".2f", annot_kws={"size": 6},
            xticklabels=corr.columns, yticklabels=corr.index, cbar=True,
            cbar_kws={"label": "Correlation", "shrink": 0.6},
        )
        ax.set_title(f"β={beta}", fontsize=11, fontweight="bold")
        ax.set_xlabel("Commodity  (Energy → Metals → Agriculture)", fontsize=8)
        ax.set_ylabel("Factor", fontsize=8)
        ax.tick_params(axis="x", rotation=90, labelsize=6)
        ax.tick_params(axis="y", rotation=0,  labelsize=8)
    year = "2013–2023" if split == "train" else "2024–2026"
    fig.suptitle(f"Factor–commodity correlation — {split} set ({year})", fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

### 9b. Top commodities per factor (train, by absolute correlation)

In [ ]:
TOP_N = 5
for arch, beta in zip(BETA_ARCHS, BETAS):
    corr = factor_commodity_corr(arch, "train")
    rows = []
    for factor in corr.index:
        top = corr.loc[factor].abs().nlargest(TOP_N)
        for rank, (com, _) in enumerate(top.items(), 1):
            rows.append({
                "factor":      factor,
                "rank":        rank,
                "commodity":   com,
                "sector":      SECTOR_MAP[com],
                "correlation": round(corr.loc[factor, com], 3),
            })
    tbl = pd.DataFrame(rows).set_index(["factor", "rank"])
    print(f"\n{'─' * 60}")
    print(f"  β={beta} — top {TOP_N} commodities per factor (train, |corr|)")
    print(f"{'─' * 60}")
    display(tbl)

### 9c. Factor–sector alignment (mean |correlation| per sector)

In [ ]:
nb = len(BETA_ARCHS)
fig, axes = plt.subplots(nb, 1, figsize=(7, 3 * nb))
if nb == 1:
    axes = [axes]

for ax, (arch, beta) in zip(axes, zip(BETA_ARCHS, BETAS)):
    corr = factor_commodity_corr(arch, "train")
    alignment = pd.DataFrame({
        sector: corr[[c for c in COMMODITIES_ORDERED if SECTOR_MAP[c] == sector]].abs().mean(axis=1)
        for sector in SECTOR_ORDER
    })
    sns.heatmap(
        alignment, ax=ax, cmap="YlOrRd", vmin=0, vmax=0.6,
        annot=True, fmt=".2f", annot_kws={"size": 9}, cbar=False,
    )
    ax.set_title(f"β={beta}", fontsize=10, fontweight="bold")
    ax.set_ylabel("Factor")
    ax.tick_params(axis="y", rotation=0)

fig.suptitle("Factor–sector alignment — mean |correlation| per sector (train)", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

### 9d. Cross-beta factor agreement

Do different beta values discover the same 5 factors? Hungarian-algorithm matching on the 5×5 absolute-correlation matrix between any two beta's factor matrices. Score near 1 = same factors found; near 0 = completely different representations.

In [ ]:
from scipy.optimize import linear_sum_assignment

def _factor_agreement(arch_a, arch_b, split="train"):
    fa = pd.read_csv(FACTORS / f"{arch_a}_{split}_factors.csv", index_col=0, parse_dates=True)
    fb = pd.read_csv(FACTORS / f"{arch_b}_{split}_factors.csv", index_col=0, parse_dates=True)
    common = fa.index.intersection(fb.index)
    fa, fb = fa.loc[common].to_numpy(), fb.loc[common].to_numpy()
    K = fa.shape[1]
    cost = np.array(
        [[abs(float(np.corrcoef(fa[:, i], fb[:, j])[0, 1])) for j in range(K)] for i in range(K)]
    )
    row, col = linear_sum_assignment(-cost)
    return float(cost[row, col].mean())

agree = pd.DataFrame(
    [[_factor_agreement(a, b) for b in BETA_ARCHS] for a in BETA_ARCHS],
    index=beta_labels, columns=beta_labels,
)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    agree, ax=ax, cmap="YlGn", vmin=0, vmax=1,
    annot=True, fmt=".2f", annot_kws={"size": 9}, square=True,
    cbar_kws={"label": "Mean optimal-matched |correlation| (5 factors)"},
)
ax.set_title("Cross-beta factor agreement (train set)", fontsize=12)
ax.tick_params(axis="x", rotation=30, labelsize=9)
ax.tick_params(axis="y", rotation=0,  labelsize=9)
fig.tight_layout()
plt.show()

### 9e. Factor autocorrelation — temporal persistence

Lag-1 (daily), lag-5 (weekly), lag-21 (monthly) autocorrelation per factor. High autocorrelation → slow macro regime signal. Near zero → fast noise. Watch for autocorrelation collapsing at high β as posterior collapse kills factor dynamics.

In [ ]:
LAGS = [1, 5, 21]

nb = len(BETA_ARCHS)
fig, axes = plt.subplots(nb, 1, figsize=(7, 3 * nb))
if nb == 1:
    axes = [axes]

for ax, (arch, beta) in zip(axes, zip(BETA_ARCHS, BETAS)):
    factors = pd.read_csv(FACTORS / f"{arch}_train_factors.csv", index_col=0, parse_dates=True)
    acorr = pd.DataFrame(
        {f"lag-{lag}d": factors.apply(lambda col: col.autocorr(lag=lag)) for lag in LAGS}
    )
    sns.heatmap(
        acorr, ax=ax, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
        annot=True, fmt=".2f", annot_kws={"size": 9}, cbar=False,
    )
    ax.set_title(f"β={beta}", fontsize=10, fontweight="bold")
    ax.set_ylabel("Factor")
    ax.tick_params(axis="y", rotation=0)

fig.suptitle("Factor autocorrelation (train) — temporal persistence", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

### 9f. Rolling factor–commodity correlation (stability)

252-day rolling correlation between each factor and its highest-correlated commodity. Stable lines = factor has a consistent economic interpretation across time. Drifting or flipping lines = the factor mixes signals, especially at high β.

In [ ]:
ROLL_WINDOW = 252

nb = len(BETA_ARCHS)
fig, axes = plt.subplots(nb, 1, figsize=(14, 3 * nb), sharex=True)
if nb == 1:
    axes = [axes]

for ax, (arch, beta) in zip(axes, zip(BETA_ARCHS, BETAS)):
    corr_full = factor_commodity_corr(arch, "train")
    factors   = pd.read_csv(FACTORS / f"{arch}_train_factors.csv", index_col=0, parse_dates=True)
    rets      = returns_raw.reindex(factors.index)
    for factor in corr_full.index:
        top_com = corr_full.loc[factor].abs().idxmax()
        rolling = factors[factor].rolling(ROLL_WINDOW).corr(rets[top_com])
        ax.plot(rolling.index, rolling, lw=0.9, alpha=0.85, label=f"{factor}↔{top_com}")
    ax.axhline(0, color="k", lw=0.5, ls="--")
    ax.set_ylim(-1, 1)
    ax.set_title(f"β={beta}", fontsize=10, fontweight="bold")
    ax.set_ylabel("Rolling corr")
    ax.legend(loc="lower right", fontsize=6, ncol=5)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

fig.suptitle(f"Rolling {ROLL_WINDOW}-day factor↔top-commodity correlation (train)", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

### 9g. Factor behaviour around market events

Factor time series with markers at key commodity market events. Compare how event sensitivity changes across beta values — high β models may show muted responses if factors are collapsed toward the prior.

In [ ]:
MARKET_EVENTS = {
    "COVID crash":      (pd.Timestamp("2020-03-20"), "red"),
    "COVID recovery":   (pd.Timestamp("2020-11-09"), "green"),
    "Russia-Ukraine":   (pd.Timestamp("2022-02-24"), "darkorange"),
    "Fed hike start":   (pd.Timestamp("2022-03-16"), "purple"),
    "Energy crisis pk": (pd.Timestamp("2022-08-26"), "steelblue"),
}

nb = len(BETA_ARCHS)
fig, axes = plt.subplots(nb, 1, figsize=(14, 3 * nb), sharex=True)
if nb == 1:
    axes = [axes]

for ax, (arch, beta) in zip(axes, zip(BETA_ARCHS, BETAS)):
    factors = pd.read_csv(FACTORS / f"{arch}_train_factors.csv", index_col=0, parse_dates=True)
    for col in factors.columns:
        ax.plot(factors.index, factors[col], lw=0.7, alpha=0.75, label=col)
    for ev_name, (ev_date, ev_col) in MARKET_EVENTS.items():
        if factors.index.min() <= ev_date <= factors.index.max():
            ax.axvline(ev_date, color=ev_col, lw=1.3, ls="--", alpha=0.9, label=ev_name)
    ax.set_title(f"β={beta}", fontsize=10, fontweight="bold")
    ax.set_ylabel("Factor value")
    ax.legend(loc="upper right", fontsize=6, ncol=5)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

fig.suptitle("Factor behaviour around market events (train 2013–2023)", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()